# BRKGA Foco — Flow Shop Scheduling

Experimentos com o brkga_scs

## 0. Setup — Instalação e Imports

In [1]:
# Instala dependencias
%pip install -q pymoo scikit-fuzzy optuna numpy pandas
#!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Dependencias instaladas.')

Dependencias instaladas.


In [2]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
ROOT = '/content/drive/MyDrive/FlowShopScheduling'

import sys, os, time, json, csv
sys.path.insert(0, ROOT)

from pathlib import Path
from typing import List, Dict, Optional, Tuple, Callable

import numpy as np
import pandas as pd
import optuna
import json

DATA_DIR = Path(ROOT) / 'data' / 'taillard'
RESULTADOS_DIR = Path(ROOT) / 'resultados_brkga_scs' / 'poucas_instancias'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

INSTANCIAS_TESTE = [
    'ta001.txt', 'ta011.txt', 'ta031.txt',
    'ta041.txt', 'ta051.txt', 'ta061.txt',
    'ta071.txt', 'ta081.txt', 'ta091.txt',
    'ta101.txt', 'ta111.txt'
    ]

print(f'ROOT: {ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Resultados em: {RESULTADOS_DIR}')

ROOT: /content/drive/MyDrive/FlowShopScheduling
DATA_DIR: /content/drive/MyDrive/FlowShopScheduling/data/taillard
Resultados em: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_scs/poucas_instancias


In [4]:
# Imports do projeto
from src.flowshop_engine import FlowShopEngine
from src.solvers.adapter import (
    rodar_brkga_scs_vm,
    rodar_brkga_scs_vr,
    rodar_brkga_scs_img,
    rodar_brkga_scs_dmn,
    SolverResult,
)

from utils.tuning_optuna import carregar_melhores_parametros

print('Imports concluídos.')

Imports concluídos.


## 1. Catálogo de Instâncias — Small and Taillard

In [5]:
def construir_catalogo_taillard(data_dir: Path) -> pd.DataFrame:
    """
    Lê os arquivos ta*.txt, extrai as dimensões e retorna um DataFrame do catálogo.
    """
    # Encontra todos os arquivos que começam com "ta" e terminam com ".txt"
    todos = sorted(data_dir.glob('ta*.txt'))
    registros = []

    for f in todos:
        # Extrai o ID numérico (ex: "ta005.txt" -> 5)
        id_instancia = int(f.stem.replace('ta', ''))

        # Dá uma "espiada" rápida no arquivo para extrair jobs e machines da Linha 2
        with open(f, 'r') as file:
            linhas = [l.strip() for l in file if l.strip()]
            partes = linhas[1].split()
            n_jobs = int(partes[0])
            n_machines = int(partes[1])
        bks = int(partes[3]) # Já extrai o BKS para termos na tabela!

        registros.append({
            'arquivo': f.name,
            'id': id_instancia,
            'n_jobs': n_jobs,
            'n_machines': n_machines,
            'bks': bks,
            'path': str(f),
            'rotulo': f'{n_jobs}x{n_machines}_id{id_instancia}'
        })

    df = pd.DataFrame(registros)
    df = df.sort_values(['n_jobs', 'n_machines', 'id']).reset_index(drop=True)
    return df



#MUDAR AQUI SE EU QUISER TESTAR OUTRO CATALOGO
catalogo = construir_catalogo_taillard(DATA_DIR)

#Catalogo com menos instancia apenas para testes rapidos
# COMENTAR AQUI SE QUISER A VERSÃO COMPLETA DO CATALOGO
catalogo = catalogo[catalogo['arquivo'].isin(INSTANCIAS_TESTE)].reset_index(drop=True)

print(f'Total de instâncias: {len(catalogo)}')
print()
print('Distribuição por tamanho (Jobs x Máquinas):')
# Agrupa para mostrar quantos arquivos temos de cada tamanho
distribuicao = catalogo.groupby(['n_jobs', 'n_machines']).size()
for (j, m), qtd in distribuicao.items():
    print(f'  {j}x{m}: {qtd} instâncias')

print('\n10 primeiras instâncias:')
print(catalogo.head(10).to_string(index=False))

Total de instâncias: 11

Distribuição por tamanho (Jobs x Máquinas):
  20x5: 1 instâncias
  20x10: 1 instâncias
  50x5: 1 instâncias
  50x10: 1 instâncias
  50x20: 1 instâncias
  100x5: 1 instâncias
  100x10: 1 instâncias
  100x20: 1 instâncias
  200x10: 1 instâncias
  200x20: 1 instâncias
  500x20: 1 instâncias

10 primeiras instâncias:
  arquivo  id  n_jobs  n_machines   bks                                                              path       rotulo
ta001.txt   1      20           5  1278 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta001.txt     20x5_id1
ta011.txt  11      20          10  1582 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta011.txt   20x10_id11
ta031.txt  31      50           5  2724 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta031.txt    50x5_id31
ta041.txt  41      50          10  3025 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta041.txt   50x10_id41
ta051.txt  51      50          20  3875 /content/drive/MyDrive/Fl

In [6]:
POP_SIZE = 100

# Geracoes para o experimento comparativo (se eu passar o t_param, automaticamente esse nao vai se ativo)
MAX_EVALS = 50000

# Sementes para o experimento
N_SEEDS = 5
SEED_INICIO = 100

# Objetivo
OBJECTIVE = 'Makespan'

## 5. Fábrica de Variantes (closures)

Cada variante é uma callable `(engine, seed) -> SolverResult`.

In [20]:
""" Isso é para nao ter que ficar passando os paramtros toda vez que chamar os alg...."""
def fabricar_variante_brkga_scs_vm() -> Callable:
    """BRKGA-SCS (Clustering Search) com parametros padrao"""
    def _run(engine, seed):
        return rodar_brkga_scs_vm(
            engine, seed=seed, max_evals=MAX_EVALS, objective=OBJECTIVE,
            t_param=60, # fator multiplicante para o tempo de cpu
            tam_pop=POP_SIZE, pe=0.2, pm=0.15, rho=0.7,
            z_limiar=1.5, votos_limiar=5,
            num_clusters_max=5, r_min=0.05, r_max=0.25,
            base_cooldown=5, gamma=1.5,
        )
    return _run

def fabricar_variante_brkga_scs_vr() -> Callable:
    def _run(engine, seed):
        return rodar_brkga_scs_vr(
            engine, seed=seed, max_evals=MAX_EVALS,
            t_param=60, # ou None se quiser parar por max_evals
            tam_pop=POP_SIZE, pe=0.15, pm=0.25, rho=0.85,
            raio_cluster=0.5, z_limiar=1.645, votos_limiar=5,
            max_hj_evals=600,
        )
    return _run

def fabricar_variante_brkga_scs_img() -> Callable:
    """BRKGA-SCS com Busca Local no Espaço de Imagem (r-NEH)"""
    def _run(engine, seed):
        return rodar_brkga_scs_img(
            engine, seed=seed, max_evals=MAX_EVALS, objective=OBJECTIVE,
            t_param=60, # ou None se quiser parar por max_evals
            )
    return _run

def fabricar_variante_brkga_scs_dmn() -> Callable:
    """BRKGA-SCS com Busca Local no Espaço de Domínio (Coordinate Descent)"""
    def _run(engine, seed):
        return rodar_brkga_scs_dmn(
            engine, seed=seed, max_evals=MAX_EVALS, objective=OBJECTIVE,
            t_param=60, # ou None se quiser parar por max_evals
        )
    return _run




#OPTUNAAAA

# Carrega os parâmetros do JSON salvo pelo Optuna
params_vm_optuna = carregar_melhores_parametros(
    RESULTADOS_DIR / 'optuna' / 'best_params_brkga_scs_vm.json'
)
params_vr_optuna = carregar_melhores_parametros(
    RESULTADOS_DIR / 'optuna' / 'best_params_brkga_scs_vr.json'
)


def fabricar_variante_brkga_scs_vm_optuna():
  def _run(engine, seed):
    return rodar_brkga_scs_vm(
        engine, seed=seed, max_evals=MAX_EVALS, objective=OBJECTIVE,
        t_param=60, # fator multiplicante para o tempo de cpu
        **params_vm_optuna,
    )
  return _run


def fabricar_variante_brkga_scs_vr_optuna():
  def _run(engine, seed):
    return rodar_brkga_scs_vr(
        engine, seed=seed, max_evals=MAX_EVALS, objective=OBJECTIVE,
        t_param=60, # fator multiplicante para o tempo de cpu
        **params_vr_optuna,
    )
  return _run

# Monta o dicionario de variantes
variantes = {
    'brkga_scs_vm': fabricar_variante_brkga_scs_vm(),
    #'brkga_scs_vr': fabricar_variante_brkga_scs_vr(),
    #'brkga_scs_img': fabricar_variante_brkga_scs_img(),
    #'brkga_scs_dmn': fabricar_variante_brkga_scs_dmn(),
    #'brkga_scs_vm_optuna': fabricar_variante_brkga_scs_vm_optuna(),
    #'brkga_scs_vr_optuna': fabricar_variante_brkga_scs_vr_optuna(),

}

print(f'Variantes disponiveis: {list(variantes.keys())}')


Variantes disponiveis: ['brkga_scs_vm']


### CSV - SALVAMENTO ETC

In [21]:
# Schema fixo do CSV
CAMPOS_CSV = [
    'instancia', 'grupo', 'n_jobs', 'n_machines',
    'metodo', 'seed',
    'makespan', 'tardiness', 'cfo_count', 'tempo_s',
    'lower_bound', 'bks', 'rpd',
]

#Quiser alterar o arquivo csv de comparacao é aqui essa merda
CSV_PATH = RESULTADOS_DIR / 'results_brkga_scs_vm_poucas_instancias_taillard.csv'
print(f'CSV de saida: {CSV_PATH}')
print(f'Campos: {CAMPOS_CSV}')

CSV de saida: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_scs/results_brkga_scs_vm_poucas_instancias_taillard.csv
Campos: ['instancia', 'grupo', 'n_jobs', 'n_machines', 'metodo', 'seed', 'makespan', 'tardiness', 'cfo_count', 'tempo_s', 'lower_bound', 'bks', 'rpd']


## 6. Experimentos — Execução e Coleta de Resultados - Loop Principal

In [22]:
print('=' * 72)
print('INICIANDO LOOP COMPARATIVO')
print(f'{len(catalogo)} instancias x {len(variantes)} variantes x {N_SEEDS} seeds')
print(f'Total: {len(catalogo) * len(variantes) * N_SEEDS} execucoes')
print('=' * 72)

t0_total = time.time()

with open(DATA_DIR / 'taillard_bks.json', 'r') as f:
    bks_db = json.load(f)


concluidos = set() # Para evitar reprocessar instâncias já concluídas
csv_header_escrito = CSV_PATH.exists() and CSV_PATH.stat().st_size > 0

if csv_header_escrito:
    df_existente = pd.read_csv(CSV_PATH)
    df_valido = df_existente.dropna(subset=['instancia', 'metodo', 'seed'])
    for _, r_old in df_valido.iterrows():
        concluidos.add((str(r_old['instancia']), str(r_old['metodo']), int(r_old['seed'])))
    print(f'[BACKUP] {len(concluidos)} execuções já concluídas e no CSV. Continuando de onde parou...')

for idx_inst, (_, row) in enumerate(catalogo.iterrows()):
    inst_label = row['rotulo']
    inst_path = row['path']
    n_jobs = row['n_jobs']
    n_mach = row['n_machines']
    grupo = "Taillard"

    # Carrega engine uma vez por instancia
    nj, nm, pt, dd = FlowShopEngine.carregar_instancia_taillard(inst_path)
    engine = FlowShopEngine(nj, nm, pt, dd)

    for metodo, fn_variante in variantes.items():
        for seed in range(SEED_INICIO, SEED_INICIO + N_SEEDS):
            if (row['arquivo'], metodo, seed) in concluidos:
                continue  # Pula execuções já concluídas

            # LOG de progresso detalhado
            print(
                f'[{idx_inst + 1}/{len(catalogo)}] '
                f'Inst: {row["arquivo"]} ({n_jobs}j x {n_mach}m, g{grupo}) '
                f'| Var: {metodo} '
                f'| Seed: {seed} ...',
                end=' '
            )

            try:
                t0_exec = time.time()
                result = fn_variante(engine, seed)
                elapsed = time.time() - t0_exec

                # Monta linha do CSV
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float(result.best_cost),
                    'tardiness': float(result.tardiness),
                    'cfo_count': int(result.n_evaluations),
                    'tempo_s': round(elapsed, 3),
                    'lower_bound': float(result.lower_bound),
                }

                # Nova logica RPD
                bks_real = bks_db.get(row['arquivo'].replace('.txt', ''))
                if bks_real is not None:
                    rpd = ((result.best_cost - bks_real) / bks_real) * 100.0
                else:
                    rpd = float('nan')
                linha['bks'] = bks_real
                linha['rpd'] = round(rpd, 3)

                print(f'OK  Makespan: {result.best_cost:.1f} ({elapsed:.2f}s)')

            except Exception as e:
                print(f'ERRO: {e}')
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float('nan'),
                    'tardiness': float('nan'),
                    'cfo_count': 0,
                    'tempo_s': float('nan'),
                    'lower_bound': float('nan'),
                    'bks': float('nan'),
                    'rpd': float('nan'),
                }

            # Escreve no CSV incremental (abre/fecha a cada linha = seguro)
            escrever_header = not csv_header_escrito
            with open(CSV_PATH, 'w' if escrever_header else 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=CAMPOS_CSV)
                if escrever_header:
                    writer.writeheader()
                    csv_header_escrito = True
                writer.writerow(linha)

elapsed_total = time.time() - t0_total
print('=' * 72)
print(f'LOOP CONCLUIDO em {elapsed_total:.1f}s ({elapsed_total/60:.1f}min)')
print(f'Resultados salvos em: {CSV_PATH}')


INICIANDO LOOP COMPARATIVO
11 instancias x 1 variantes x 5 seeds
Total: 55 execucoes
[BACKUP] 55 execuções já concluídas e no CSV. Continuando de onde parou...
[11/11] Inst: ta111.txt (500j x 20m, gTaillard) | Var: brkga_scs_vm | Seed: 100 ... OK  Makespan: 27681.0 (300.68s)
[11/11] Inst: ta111.txt (500j x 20m, gTaillard) | Var: brkga_scs_vm | Seed: 101 ... OK  Makespan: 27453.0 (300.57s)
[11/11] Inst: ta111.txt (500j x 20m, gTaillard) | Var: brkga_scs_vm | Seed: 102 ... OK  Makespan: 27460.0 (300.40s)
[11/11] Inst: ta111.txt (500j x 20m, gTaillard) | Var: brkga_scs_vm | Seed: 103 ... OK  Makespan: 27434.0 (300.46s)
[11/11] Inst: ta111.txt (500j x 20m, gTaillard) | Var: brkga_scs_vm | Seed: 104 ... OK  Makespan: 27589.0 (300.56s)
LOOP CONCLUIDO em 1502.7s (25.0min)
Resultados salvos em: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_scs/results_brkga_scs_vm_poucas_instancias_taillard.csv


## 7. Resumo Final

In [27]:
# 1. Lista com os CSVs dos experimentos que você quer comparar
ARQUIVOS_CSV = [
    RESULTADOS_DIR / 'results_brkga_scs_vm_poucas_instancias_taillard.csv',
    RESULTADOS_DIR / 'results_brkga_scs_vr_poucas_instancias_taillard.csv',
    RESULTADOS_DIR / 'results_brkga_scs_img_poucas_instancias_taillard.csv',
    RESULTADOS_DIR / 'results_brkga_scs_dmn_poucas_instancias_taillard.csv',
]

# Caminho onde será salvo o novo CSV consolidado (médias por instância x método)
CSV_RESUMO_PATH = RESULTADOS_DIR / 'resumo_comparativo_por_instancia.csv'
CSV_RESUMO_GLOBAL_PATH = RESULTADOS_DIR / 'resumo_comparativo_global.csv'

# 2. Carrega e junta todos os CSVs da lista que existirem
dfs = []
for caminho in ARQUIVOS_CSV:
    if caminho.exists():
        dfs.append(pd.read_csv(caminho))
    else:
        print(f'[AVISO] Arquivo não encontrado (ignorado): {caminho.name}')

if not dfs:
    raise FileNotFoundError('Nenhum dos arquivos CSV informados foi encontrado!')

df_total = pd.concat(dfs, ignore_index=True)
print(f'Total de execuções carregadas ({len(dfs)} arquivos): {len(df_total)} linhas\n')

# 3. Agrupa por Instância + Método (calculando a média das seeds)
agg_instancia = df_total.groupby(['instancia', 'n_jobs', 'n_machines', 'metodo'], as_index=False).agg(
    makespan_medio=('makespan', 'mean'),
    tardiness_medio=('tardiness', 'mean'),
    desvio=('makespan', 'std'),
    melhor=('makespan', 'min'),
    pior=('makespan', 'max'),
    tempo_medio=('tempo_s', 'mean'),
    bks=('bks', 'first'),
    rpd_medio=('rpd', 'mean'),
    lower_bound=('lower_bound', 'first'),
)

# Ordena por tamanho da instância (jobs, máquinas), nome da instância e método
agg_instancia = agg_instancia.sort_values(['n_jobs', 'n_machines', 'instancia', 'metodo']).reset_index(drop=True)

# Arredondamento das colunas numéricas
agg_instancia['makespan_medio'] = agg_instancia['makespan_medio'].round(1)
agg_instancia['tardiness_medio'] = agg_instancia['tardiness_medio'].round(1)
agg_instancia['desvio'] = agg_instancia['desvio'].round(1)
agg_instancia['tempo_medio'] = agg_instancia['tempo_medio'].round(3)
agg_instancia['rpd_medio'] = agg_instancia['rpd_medio'].round(2)

# Salva o CSV detalhado por instância
agg_instancia.to_csv(CSV_RESUMO_PATH, index=False)

print('=' * 130)
print('TABELA COMPARATIVA — Média das Seeds por Instância x Método')
print('=' * 130)
print(agg_instancia.to_string(index=False))
print()

# 4. Resumo Global (agregando todas as instâncias por Método)
global_agg = df_total.groupby('metodo', as_index=False).agg(
    makespan_medio=('makespan', 'mean'),
    tardiness_medio=('tardiness', 'mean'),
    tempo_medio=('tempo_s', 'mean'),
    rpd_medio=('rpd', 'mean'),
)

global_agg['makespan_medio'] = global_agg['makespan_medio'].round(1)
global_agg['tardiness_medio'] = global_agg['tardiness_medio'].round(1)
global_agg['tempo_medio'] = global_agg['tempo_medio'].round(3)
global_agg['rpd_medio'] = global_agg['rpd_medio'].round(2)

# Salva também o resumo global em CSV
global_agg.to_csv(CSV_RESUMO_GLOBAL_PATH, index=False)

print('-' * 80)
print('RESUMO GLOBAL — Média Geral por Método')
print('-' * 80)
print(global_agg.to_string(index=False))
print()
print(f'CSV detalhado (por instância) salvo em: {CSV_RESUMO_PATH}')
print(f'CSV resumo global salvo em:             {CSV_RESUMO_GLOBAL_PATH}')

Total de execuções carregadas (4 arquivos): 220 linhas

TABELA COMPARATIVA — Média das Seeds por Instância x Método
instancia  n_jobs  n_machines        metodo  makespan_medio  tardiness_medio  desvio  tempo_medio   bks  rpd_medio  lower_bound
ta001.txt      20           5 brkga_scs_dmn          1297.0          15554.4     0.0        3.079  1278       1.49       1249.0
ta001.txt      20           5 brkga_scs_img          1297.0          15969.0     0.0        3.143  1278       1.49       1249.0
ta001.txt      20           5  brkga_scs_vm          1298.6          16167.6     3.6        3.008  1278       1.61       1249.0
ta001.txt      20           5  brkga_scs_vr          1307.0          15887.4    13.7        3.144  1278       2.27       1249.0
ta011.txt      20          10 brkga_scs_dmn          1627.8          22834.8    19.1        6.011  1582       2.90       1479.0
ta011.txt      20          10 brkga_scs_img          1627.0          23026.8     8.3        6.016  1582       2.84  